In [1]:
from dataclasses import dataclass, field
from PIL import Image

import numpy as np
from numpy.typing import NDArray
from typing import Iterator

import struct
from pathlib import Path
import json
import datetime
import io

In [2]:
# Image format codec mapping
IMAGE_FORMAT_CODES = {
    100: "monoraw",   # Grayscale uncompressed
    200: "raw",       # Color BGR uncompressed
    101: "brgb8",     # Bayer pattern raw
    102: "monojpg",   # Grayscale JPEG compressed
    201: "jpg",       # Color JPEG compressed
    103: "jbrgb",     # Bayer JPEG compressed
    1: "monopng",     # Grayscale PNG compressed
    2: "png",         # Color PNG compressed
}

# Codec name aliases
CODEC_ALIASES = {
    "monoraw": "imageFormat100",
    "raw": "imageFormat200",
    "brgb8": "imageFormat101",
    "monojpg": "imageFormat102",
    "jpg": "imageFormat201",
    "jbrgb": "imageFormat103",
    "monopng": "imageFormat001",
    "png": "imageFormat002",
}

# Which codecs use compression (variable-length frames)
COMPRESSED_CODECS = {"monojpg", "jpg", "jbrgb", "monopng", "png"}
UNCOMPRESSED_CODECS = {"monoraw", "raw", "brgb8"}

HEADER_SIZE = 1024
MAGIC = 0xFEED

## SeqHeader
Read and return the fields within the .seq header

In [7]:
@dataclass
class SeqHeader:
    """Parsed header of a .seq file."""

    magic: int = MAGIC
    name: str = "Norpix seq"
    version: int = 0
    header_size: int = HEADER_SIZE
    description: str = ""
    width: int = 0
    height: int = 0
    bit_depth: int = 8
    bit_depth_real: int = 8
    image_size_bytes: int = 0
    image_format: int = 100
    num_frames: int = 0
    true_image_size: int = 0
    fps: float = 30.0
    codec: str = ""

    @property
    def codec_name(self) -> str:
        """Human-readable codec name."""
        return IMAGE_FORMAT_CODES.get(self.image_format, f"unknown({self.image_format})")

    @property
    def is_compressed(self) -> bool:
        """Whether frames use variable-length compression."""
        return self.codec_name in COMPRESSED_CODECS

    @property
    def num_channels(self) -> int:
        """Number of color channels."""
        return self.bit_depth // (self.bit_depth_real or 8)

    @classmethod
    def from_file(cls, f) -> SeqHeader:
        """Read and parse the header from an open file handle."""
        f.seek(0)
        raw = f.read(HEADER_SIZE)
        if len(raw) < HEADER_SIZE:
            raise ValueError("File too small to contain a valid .seq header")

        # Magic number (bytes 0-3)
        magic = struct.unpack_from("<I", raw, 0)[0]
        if magic != MAGIC:
            raise ValueError(f"Invalid .seq magic: 0x{magic:08X} (expected 0x{MAGIC:08X})")

        # Name string (bytes 4-23, 10 uint16 chars)
        name_chars = struct.unpack_from("<10H", raw, 4)
        name = "".join(chr(c) for c in name_chars if 0 < c < 128).strip()

        # 4 bytes padding (24-27)

        # Version and header size (bytes 28-35)
        version, header_size = struct.unpack_from("<iI", raw, 28)

        # Description (bytes 36-547, 256 uint16 chars)
        desc_chars = struct.unpack_from("<256H", raw, 36)
        description = "".join(chr(c) for c in desc_chars if 0 < c < 128).strip()

        # 9 uint32 fields (bytes 548-583)
        fields = struct.unpack_from("<9I", raw, 548)
        width = fields[0]
        height = fields[1]
        bit_depth = fields[2]
        bit_depth_real = fields[3]
        image_size_bytes = fields[4]
        image_format = fields[5]
        num_frames = fields[6]
        # fields[7] is reserved (should be 0)
        true_image_size = fields[8]

        # Frame rate (bytes 584-591)
        fps = struct.unpack_from("<d", raw, 584)[0]

        codec = f"imageFormat{image_format:03d}"

        return cls(
            magic=magic,
            name=name,
            version=version,
            header_size=header_size,
            description=description,
            width=width,
            height=height,
            bit_depth=bit_depth,
            bit_depth_real=bit_depth_real,
            image_size_bytes=image_size_bytes,
            image_format=image_format,
            num_frames=num_frames,
            true_image_size=true_image_size,
            fps=fps,
            codec=codec,
        )


## SeqIndex
Determining the frames positions

In [9]:
@dataclass
class SeqIndex:
    """Frame seek index for a .seq file.

    For uncompressed formats, frame offsets are computed arithmetically.
    For compressed formats, a seek table must be built by scanning the file.
    """

    offsets: list[int] = field(default_factory=list)
    num_frames: int = 0
    timestamp_size: int = 8  # 8 bytes for version >= 5, 6 for older

    def frame_offset(self, frame: int) -> int:
        """Get the byte offset for a given frame number."""
        if frame < 0 or frame >= self.num_frames:
            raise IndexError(f"Frame {frame} out of range [0, {self.num_frames})")
        return self.offsets[frame]

    def save(self, path: str | Path) -> None:
        """Save the seek index to a JSON file."""
        data = {
            "num_frames": self.num_frames,
            "timestamp_size": self.timestamp_size,
            "offsets": self.offsets,
        }
        with open(path, "w") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str | Path) -> SeqIndex:
        """Load a seek index from a JSON file."""
        with open(path) as f:
            data = json.load(f)
        return cls(
            offsets=data["offsets"],
            num_frames=data["num_frames"],
            timestamp_size=data.get("timestamp_size", 8),
        )

    @classmethod
    def build_uncompressed(cls, header: SeqHeader) -> SeqIndex:
        """Build index for uncompressed formats (constant frame stride)."""
        offsets = [
            HEADER_SIZE + i * header.true_image_size
            for i in range(header.num_frames)
        ]
        return cls(
            offsets=offsets,
            num_frames=header.num_frames,
            timestamp_size=8 if header.version >= 5 else 6,
        )

    @classmethod
    def build_compressed(cls, f, header: SeqHeader) -> SeqIndex:
        """Build index for compressed formats by scanning the file.

        Scans frame-by-frame using the 4-byte size field at each frame start.
        After the first frame, auto-detects the padding between frames by
        searching for the next valid frame header (a size field followed by
        a recognizable image magic byte sequence).

        Args:
            f: Open file handle in binary read mode.
            header: Parsed SeqHeader.

        Returns:
            SeqIndex with computed frame offsets.
        """
        file_size = f.seek(0, 2)
        n_max = header.num_frames if header.num_frames > 0 else 10_000_000

        ts_size = 8 if header.version >= 5 else 6
        extra = None  # Will be auto-detected from first frame transition

        # Image magic bytes that signal the start of a frame's image data
        # (right after the 4-byte size field)
        JPEG_SOI = b"\xff\xd8"
        PNG_SIG = b"\x89\x50"  # first 2 bytes of PNG signature

        offsets = [HEADER_SIZE]

        for i in range(1, n_max):
            prev = offsets[i - 1]
            f.seek(prev)
            size_bytes = f.read(4)
            if len(size_bytes) < 4:
                break
            frame_size = struct.unpack("<I", size_bytes)[0]
            if frame_size == 0 or frame_size > file_size:
                break

            if extra is not None:
                # We know the padding; jump directly
                next_offset = prev + frame_size + extra
            else:
                # Auto-detect: scan forward from end of frame data to find
                # the next valid size + image-magic pair.
                # The gap is typically ts_size + 0..16 bytes of padding.
                search_start = prev + frame_size + ts_size
                found = False
                for pad in range(0, 32, 2):  # try even paddings up to 30
                    candidate = search_start + pad
                    if candidate + 6 > file_size:
                        break
                    f.seek(candidate)
                    probe = f.read(6)
                    if len(probe) < 6:
                        break
                    cand_size = struct.unpack("<I", probe[:4])[0]
                    cand_magic = probe[4:6]
                    if (0 < cand_size < file_size
                            and cand_magic in (JPEG_SOI, PNG_SIG)):
                        extra = frame_size + ts_size + pad - frame_size  # = ts_size + pad
                        extra = ts_size + pad
                        next_offset = candidate
                        found = True
                        break
                if not found:
                    break

            if next_offset >= file_size:
                break

            # Final validation
            f.seek(next_offset)
            check = f.read(6)
            if len(check) < 6:
                break
            check_size = struct.unpack("<I", check[:4])[0]
            if check_size == 0 or check_size > file_size:
                break

            offsets.append(next_offset)

        return cls(
            offsets=offsets,
            num_frames=len(offsets),
            timestamp_size=ts_size,
        )


## SeqReader

In [ ]:
class SeqReader:
    def __init__(self, path: str | Path):
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"File not found: {self.path}")

        self._file = open(self.path, "rb")
        self.header = SeqHeader.from_file(self._file)
        self.index = self._load_or_build_index()

        # Recompute FPS from timestamps (MATLAB does this)
        self._recompute_fps()

    def _load_or_build_index(self) -> SeqIndex:
        """Load cached index or build one by scanning the file."""
        if not self.header.is_compressed:
            return SeqIndex.build_uncompressed(self.header)

        # Check for cached index
        cache_path = self.path.with_suffix(".seq-index.json")
        if cache_path.exists():
            try:
                idx = SeqIndex.load(cache_path)
                if idx.num_frames > 0:
                    return idx
            except (json.JSONDecodeError, KeyError):
                pass

        # Build by scanning
        idx = SeqIndex.build_compressed(self._file, self.header)

        # Cache for next time
        try:
            idx.save(cache_path)
        except OSError:
            pass

        return idx

    def _recompute_fps(self) -> None:
        """Recompute FPS from actual timestamps (first 100 frames)."""
        try:
            n = min(100, len(self))
            if n < 2:
                return
            ts = np.array([self.get_timestamp_float(i) for i in range(n)])
            ds = np.diff(ts)
            median_ds = np.median(ds)
            # Filter outliers
            ds = ds[np.abs(ds - median_ds) < 0.005]
            if len(ds) > 0:
                computed = 1.0 / np.mean(ds)
                if np.isfinite(computed) and computed >= 1.0:
                    self.header.fps = float(computed)
        except Exception:
            pass  # Keep the header FPS as-is

    @property
    def num_frames(self) -> int:
        return self.index.num_frames

    def _read_raw_frame(self, frame: int) -> tuple[bytes, int]:
        """Read raw frame bytes and return (data_bytes, offset_after_data).

        For uncompressed: reads image_size_bytes of raw pixel data.
        For compressed: reads the size field, then (size-4) bytes of encoded data.

        Returns:
            Tuple of (raw_bytes, file_position_after_image_data).
        """
        offset = self.index.frame_offset(frame)
        self._file.seek(offset)

        if self.header.is_compressed:
            nbytes = struct.unpack("<I", self._file.read(4))[0]
            data = self._file.read(nbytes - 4)
            return data, self._file.tell()
        else:
            data = self._file.read(self.header.image_size_bytes)
            return data, self._file.tell()

    def get_frame(self, frame: int) -> NDArray[np.uint8]:
        """Read and decode a single frame.

        Args:
            frame: Frame index (0-based). Negative indices count from the end.

        Returns:
            Decoded image as numpy array with shape (height, width) for grayscale
            or (height, width, 3) for color (RGB).
        """
        if frame < 0:
            frame = self.num_frames + frame
        if frame < 0 or frame >= self.num_frames:
            raise IndexError(f"Frame {frame} out of range [0, {self.num_frames})")

        data, _ = self._read_raw_frame(frame)
        return self._decode_frame(data)

    def _decode_frame(self, data: bytes) -> NDArray[np.uint8]:
        """Decode raw frame bytes into a numpy image array."""
        codec = self.header.codec_name

        if codec in ("monoraw", "raw", "brgb8"):
            return self._decode_raw(data)
        elif codec in ("monojpg", "jpg", "jbrgb"):
            return self._decode_jpeg(data)
        elif codec in ("monopng", "png"):
            return self._decode_png(data)
        else:
            raise ValueError(f"Unsupported codec: {codec}")

    def _decode_raw(self, data: bytes) -> NDArray[np.uint8]:
        """Decode uncompressed raw pixel data."""
        h, w = self.header.height, self.header.width
        nch = self.header.num_channels
        arr = np.frombuffer(data, dtype=np.uint8)

        if nch == 1:
            return arr.reshape(h, w)
        else:
            # Data stored as BGR interleaved
            arr = arr.reshape(h, w, nch)
            # Convert BGR -> RGB
            return arr[:, :, ::-1].copy()

    def _decode_jpeg(self, data: bytes) -> NDArray[np.uint8]:
        """Decode JPEG compressed frame."""
        if Image is None:
            raise ImportError("Pillow is required for JPEG decoding: pip install Pillow")
        img = Image.open(io.BytesIO(data))
        return np.array(img)

    def _decode_png(self, data: bytes) -> NDArray[np.uint8]:
        """Decode PNG compressed frame."""
        if Image is None:
            raise ImportError("Pillow is required for PNG decoding: pip install Pillow")
        img = Image.open(io.BytesIO(data))
        return np.array(img)

    def get_timestamp(self, frame: int) -> datetime.datetime:
        """Get the timestamp for a frame as a datetime object.

        Note: .seq files store local time, not UTC.
        """
        tfloat = self.get_timestamp_float(frame)
        return datetime.datetime.fromtimestamp(tfloat)

    def get_timestamp_float(self, frame: int) -> float:
        """Get the timestamp for a frame as seconds since epoch."""
        if frame < 0:
            frame = self.num_frames + frame
        if frame < 0 or frame >= self.num_frames:
            raise IndexError(f"Frame {frame} out of range [0, {self.num_frames})")

        _, pos_after_data = self._read_raw_frame(frame)
        self._file.seek(pos_after_data)

        ts_sec = struct.unpack("<I", self._file.read(4))[0]
        ts_ms = struct.unpack("<H", self._file.read(2))[0]

        result = ts_sec + ts_ms / 1000.0

        if self.index.timestamp_size == 8:
            ts_us = struct.unpack("<H", self._file.read(2))[0]
            result += ts_us / 1_000_000.0

        return result

    def get_timestamps_float(self) -> NDArray[np.float64]:
        """Get all frame timestamps as an array of seconds since epoch."""
        return np.array([self.get_timestamp_float(i) for i in range(self.num_frames)])

    def get_frame_with_timestamp(
        self, frame: int
    ) -> tuple[NDArray[np.uint8], float]:
        """Read a frame and its timestamp in a single seek operation."""
        if frame < 0:
            frame = self.num_frames + frame
        if frame < 0 or frame >= self.num_frames:
            raise IndexError(f"Frame {frame} out of range [0, {self.num_frames})")

        data, pos = self._read_raw_frame(frame)
        img = self._decode_frame(data)

        self._file.seek(pos)
        ts_sec = struct.unpack("<I", self._file.read(4))[0]
        ts_ms = struct.unpack("<H", self._file.read(2))[0]
        ts = ts_sec + ts_ms / 1000.0
        if self.index.timestamp_size == 8:
            ts_us = struct.unpack("<H", self._file.read(2))[0]
            ts += ts_us / 1_000_000.0

        return img, ts

    # def to_mp4(
    #     self,
    #     output: str | Path | None = None,
    #     *,
    #     fps: float | None = None,
    #     codec: str = "libx264",
    #     crf: int = 25,
    #     preset: str = "superfast",
    #     pixelformat: str = "yuv420p",
    #     frame_range: tuple[int, int] | None = None,
    #     output_params: list[str] | None = None,
    # ) -> tuple[Path, list[int]]:
    #     """Re-encode the .seq file to MP4 using ffmpeg (via sleap-io's VideoWriter).

    #     Corrupt frames are replaced with a copy of the previous good frame
    #     (or a black frame if no previous frame exists).

    #     Args:
    #         output: Output MP4 path. Defaults to same name with .mp4 extension.
    #         fps: Output frame rate. Defaults to the source FPS.
    #         codec: Video codec (e.g., "libx264", "libx265"). Default: "libx264".
    #         crf: Constant rate factor (2-51, lower = higher quality). Default: 25.
    #         preset: Encoding speed preset (ultrafast, superfast, veryfast, faster,
    #             fast, medium, slow, slower, veryslow). Default: "superfast".
    #         pixelformat: Pixel format for output. Default: "yuv420p".
    #         frame_range: Optional (start, end) tuple for a subset of frames.
    #             Both indices are 0-based, end is exclusive.
    #         output_params: Additional ffmpeg output parameters as a list of strings.

    #     Returns:
    #         Tuple of (output_path, list_of_corrupt_frame_indices).
    #     """
    #     try:
    #         from sleap_io.io.video_writing import VideoWriter
    #     except ImportError:
    #         raise ImportError(
    #             "sleap-io is required for MP4 encoding: "
    #             "uv pip install sleap-io"
    #         )

    #     if output is None:
    #         output = self.path.with_suffix(".mp4")
    #     output = Path(output)

    #     if fps is None:
    #         fps = self.header.fps
    #     if fps < 1.0:
    #         fps = 30.0

    #     start = 0
    #     end = self.num_frames
    #     if frame_range is not None:
    #         start, end = frame_range
    #         start = max(0, start)
    #         end = min(end, self.num_frames)

    #     total = end - start
    #     is_grayscale = self.header.num_channels == 1

    #     writer = VideoWriter(
    #         filename=output,
    #         fps=fps,
    #         codec=codec,
    #         crf=crf,
    #         preset=preset,
    #         pixelformat=pixelformat,
    #         output_params=output_params or [],
    #     )

    #     import time

    #     corrupt_frames: list[int] = []
    #     prev_frame = None
    #     t0 = time.perf_counter()
    #     with writer:
    #         for i in range(start, end):
    #             try:
    #                 frame = self.get_frame(i)
    #             except Exception:
    #                 corrupt_frames.append(i)
    #                 if prev_frame is not None:
    #                     frame = prev_frame
    #                 else:
    #                     frame = np.zeros(
    #                         (self.header.height, self.header.width), dtype=np.uint8
    #                     )
    #             # h264 with yuv420p requires 3-channel input
    #             if frame.ndim == 2:
    #                 frame = np.stack([frame, frame, frame], axis=-1)
    #             writer(frame)
    #             prev_frame = frame

    #             done = i - start + 1
    #             if done % 5000 == 0 or done == total:
    #                 elapsed = time.perf_counter() - t0
    #                 speed = done / elapsed
    #                 eta = (total - done) / speed if speed > 0 else 0
    #                 print(
    #                     f"  [{done}/{total}] "
    #                     f"{speed:.0f} fps, "
    #                     f"ETA {eta:.0f}s"
    #                 )

    #     elapsed = time.perf_counter() - t0
    #     if corrupt_frames:
    #         print(
    #             f"  WARNING: {len(corrupt_frames)} corrupt frame(s) replaced with previous frame"
    #         )
    #     print(
    #         f"Wrote {total} frames to {output} "
    #         f"in {elapsed:.1f}s ({total / elapsed:.0f} fps)"
    #     )
    #     return output, corrupt_frames

    def __len__(self) -> int:
        return self.num_frames

    def __getitem__(self, idx: int | slice) -> NDArray[np.uint8]:
        if isinstance(idx, slice):
            indices = range(*idx.indices(self.num_frames))
            return np.stack([self.get_frame(i) for i in indices])
        return self.get_frame(idx)

    def __iter__(self) -> Iterator[NDArray[np.uint8]]:
        for i in range(self.num_frames):
            yield self.get_frame(i)

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

    def close(self) -> None:
        """Close the underlying file handle."""
        if hasattr(self, "_file") and self._file and not self._file.closed:
            self._file.close()

    def __del__(self):
        self.close()

    def __repr__(self) -> str:
        return (
            f"SeqReader('{self.path.name}')\n"
            f"  Frames: {self.num_frames}\n"
            f"  Size: {self.header.width}x{self.header.height}\n"
            f"  Codec: {self.header.codec_name} ({self.header.codec})\n"
            f"  FPS: {self.header.fps:.2f}\n"
            f"  Bit depth: {self.header.bit_depth_real}"
        )


# Test

In [ ]:
reader = SeqReader("olympic/basketball_layup/_CUsxbE0nqU_00106_00235.seq")
print(reader.header)
frame = reader.get_frame(0)
ts = reader.get_timestamp(0)

print(ts)
reader.close()

SeqHeader(magic=65261, name='Norpix seq', version=3, header_size=1024, description='No Description', width=480, height=270, bit_depth=24, bit_depth_real=8, image_size_bytes=388800, image_format=201, num_frames=130, true_image_size=389120, fps=23.277686339054785, codec='imageFormat201')
1970-01-01 10:00:00
